# TP6 : Scrapping de News financières

**Objectif :** Pour chaque entreprise, récupérer les dernières actualités à partir de NewsAPI, les filtrer
par nom de l’entreprise, et stocker les articles pertinents (titre, description, source, date de publication)
dans un fichier JSON.

In [2]:
import requests
import json
from datetime import datetime, timedelta
import os
import pandas as pd

## 1.1 Obtention d'une clé NewsAPI

NewAPI est une API de scrapping d’articles de presse permettant d’accéder à des centaines de sources
internationales fiables.

Elle permet notamment de :
- Rechercher des articles en fonction de mots-clés (ex: nom d’une entreprise)
- Filtrer par date de publication, langue, ou source d’information
- Obtenir pour chaque article : le titre, la description, la date, la source et l’URL.

**Créer un compte News API et récupérer sa clé personnelle (dans le tableau de bord)**

API Key : 8df5fe70ce6c47f799dbb257de5c2821

**Remarque :** La version gratuite permet un usage limité (100 requêtes/jour et 100 articles/requête).

Pour entrainement et mise en production d’un modèle, il faut un scrapping quotidien.

## 1.2 Scrapping de News

Création d'une fonction pour scrapping et sauvegarde de News

### Test

In [18]:
company_name = "Apple"
url = 'https://newsapi.org/v2/everything'
last_day = datetime.today().strftime('%Y-%m-%d')
first_day = datetime.today()- timedelta(days=10)
news_dict = {}
api_key = "8df5fe70ce6c47f799dbb257de5c2821"

params = {
"sources": 'financial-post, the-wall-street-journal, bloomberg, the-washington-post, australian-financial-review, bbc-news, cnn',
"q": company_name,
"apiKey": api_key,
"language": "en",
"pageSize": 100,
"from": first_day,
"to": last_day,
}

response = requests.get(url, params=params)
if response.status_code == 200:
    data = response.json()

In [20]:
data

{'status': 'ok',
 'totalResults': 12,
 'articles': [{'source': {'id': 'financial-post', 'name': 'Financial Post'},
   'author': 'Bloomberg News',
   'title': 'Dangote Seeks More Nigerian Crude to Cushion Fuel-Price Gains',
   'description': 'Billionaire Aliko Dangote’s refinery is seeking to buy more crude from Nigeria’s government to help soften the impact of rising fuel costs.',
   'url': 'https://financialpost.com/pmn/business-pmn/dangote-seeks-more-nigerian-crude-to-cushion-fuel-price-gains',
   'urlToImage': 'https://smartcdn.gprod.postmedia.digital/financialpost/wp-content/uploads/2026/03/776213099.jpg',
   'publishedAt': '2026-03-09T16:42:03Z',
   'content': '(Bloomberg) Billionaire Aliko Dangotes refinery is seeking to buy more crude from Nigerias government to help soften the impact of rising fuel costs.\r\nTHIS CONTENT IS RESERVED FOR SUBSCRIBERS ONLY\r\nS… [+4994 chars]'},
  {'source': {'id': 'financial-post', 'name': 'Financial Post'},
   'author': 'Bloomberg News',
   'tit

In [15]:
data['articles'][0]

{'source': {'id': 'financial-post', 'name': 'Financial Post'},
 'author': 'Bloomberg News',
 'title': 'Dangote Seeks More Nigerian Crude to Cushion Fuel-Price Gains',
 'description': 'Billionaire Aliko Dangote’s refinery is seeking to buy more crude from Nigeria’s government to help soften the impact of rising fuel costs.',
 'url': 'https://financialpost.com/pmn/business-pmn/dangote-seeks-more-nigerian-crude-to-cushion-fuel-price-gains',
 'urlToImage': 'https://smartcdn.gprod.postmedia.digital/financialpost/wp-content/uploads/2026/03/776213099.jpg',
 'publishedAt': '2026-03-09T16:42:03Z',
 'content': '(Bloomberg) Billionaire Aliko Dangotes refinery is seeking to buy more crude from Nigerias government to help soften the impact of rising fuel costs.\r\nTHIS CONTENT IS RESERVED FOR SUBSCRIBERS ONLY\r\nS… [+4994 chars]'}

### Fonction pipeline

In [3]:
NEWS_DIR = "companies_news"
API_KEY = "8df5fe70ce6c47f799dbb257de5c2821"
SOURCES = 'financial-post, the-wall-street-journal, bloomberg, the-washington-post, australian-financial-review, bbc-news, cnn'
PAGE_SIZE = 100

In [4]:
def load_existing_data(company_name):
    filename = f"{company_name}_news.json"
    if os.path.exists(filename):
        with open(filename, 'r') as file:
            return json.load(file)
    return None

In [5]:
def get_news_by_date(company_name) :
    company_lower = company_name.lower()
    url = 'https://newsapi.org/v2/everything'
    last_day = datetime.today().strftime('%Y-%m-%d')
    first_day = datetime.today()- timedelta(days=15)

    params = {
    "sources": SOURCES,
    "q": company_name,
    "apiKey": API_KEY,
    "language": "en",
    "pageSize": PAGE_SIZE,
    "from": first_day,
    "to": last_day,
    }

    news_dict = load_existing_data(company_name) or {}
    
    response = requests.get(url, params=params)
    if response.status_code == 200:
        data = response.json()
        for article in data['articles']:
            title = article['title']
            description = article['description']

            # Filtrer les articles qui ne mentionnent pas l'entreprise
            if company_lower not in title.lower() and company_lower not in description.lower():
                continue

            date = article['publishedAt'].split('T')[0]
            # TODO : vérifier à quoi sert cette ligne 
            existing_titles = {a['title'] for a in news_dict.get(date, [])}
            
            if title in existing_titles:
                continue
            
            news_dict.setdefault(date, []).append({
                "title": title,
                "description": description,
                "source": article['source']['name'],
                "url": article['url'],
                "publishedAt": article['publishedAt']
            })

    # Sauvegarder les données dans un fichier JSON
    with open(f'{NEWS_DIR}/{company_name}_news.json', 'w') as f:
        json.dump(news_dict, f, indent=4)

    return news_dict

In [6]:
company_name = "JD"
get_news_by_date(company_name)

{'2026-03-18': [{'title': 'Vance to Huddle With Oil Executives Amid Spiking Gasoline Prices',
   'description': 'Vice President JD Vance and other key Trump administration officials plan to huddle with oil executives Thursday as the White House looks for ways to tame surging fuel prices after the US attack on Iran, according to people familiar with the matter.',
   'source': 'Financial Post',
   'url': 'https://financialpost.com/pmn/business-pmn/vance-to-huddle-with-oil-executives-amid-spiking-gasoline-prices',
   'publishedAt': '2026-03-18T16:33:12Z'}]}